In [ ]:
# This file will handle new incoming data for orders
# Saving the original data to the bronze table then procesing it through staging stages within a bronze and silve staging table
# New data is then merged with the existing data that resides in the gold.fact_orders

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [ ]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3:sports-consolidated-dp/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed/'

bronze_table = f'{catalog}.{bronze_schema}.{data_source}'
silver_table = f'{catalog}.{silver_schema}.{data_source}'



In [ ]:
# Read from the 'landing' folder from S3 bucket and forward it into the staging
# Landing folder is new incoming data to be appended to the Bronze
df = spark.read.options(header=True, inferSchema=True).csv(f'{landing_path}/*.csv').withColumn('read_timestamp', F.current_timestamp()), '_metadata.file_name', '_metadata.file_size'

print('Total Rows: ', df.count())
df.show(5)
# Total Rows:  8834
# +-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |     order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
# +-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |FDEC818102602|Tuesday, December...|     789102|  25891502|    196.0|2025-11-24 14:09:...|orders_2025_12_16...|    23060|
# |FDEC818102602|Tuesday, December...|     789102|  25891503|     NULL|2025-11-24 14:09:...|orders_2025_12_16...|    23060|
# |FDEC818102602|Tuesday, December...|     789102|  25891602|    147.0|2025-11-24 14:09:...|orders_2025_12_16...|    23060|
# |FDEC818102602|Tuesday, December...|     789102|  25891101|    337.0|2025-11-24 14:09:...|orders_2025_12_16...|    23060|
# |FDEC818102602|Tuesday, December...|    INVALID|  25891202|    202.0|2025-11-24 14:09:...|orders_2025_12_16...|    23060|
# +-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# only showing top 5 rows

In [ ]:
# Append new incoming data along with existing data to bronze table
df.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('append') \
    .saveAsTable(bronze_table)
    

In [ ]:
# Staging table to process the new ariving incremental data
df.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{bronze_schema}.staging_{data_source}')

In [ ]:
# Moving files from source to processed directory
files = dbutils.fs.ls(landing_path)
for file in files:
    dbutils.fs.mv(
        file.path,
        f'{processed_path}/{file.name}',
        True
    )

In [ ]:
# ---> SILVER <---
df_orders = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source};')
df_orders.show(2)

# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |FDEC85903601|Thursday, Decembe...|     789903|  77777777|    342.0|2025-11-24 14:09:...|orders_2025_12_04...|    22385|
# |FDEC85903601|Thursday, Decembe...|     789903|  25891302|     53.0|2025-11-24 14:09:...|orders_2025_12_04...|    22385|
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# only showing top 2 rows

In [ ]:
# Keep rows where order_qty is present
df_orders = df.orders.filter(F.col('order_qty').isNotNull())

# Clean customer_id: keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    'customer_id',
    F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id'))
     .otherwise('999999')
     .cast('string')
)

# Remove weekday name from the date text
# 'Tuesday, July 01, 2025' -> 'July 01, 2025
df_orders = df_orders.withColumn(
    'order_placement_date',
    F.regexp_replace(F.col('order_placement_date'), r'^[A-Za-z]+,\s*')
)

# Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    'order_placement_date',
    F.coalesce(
        F.try_to_date('order_placement_date', 'yyyy/MM/dd'),
        F.try_to_date('order_placement_date', 'dd-MM-yyyy'),
        F.try_to_date('order_placement_date', 'dd/MM/yyyy'),
        F.try_to_date('order_placement_date', 'MMMM dd, yyyy')
    )
)

# Drop duplicates
df_orders = df_orders.dropDuplicates(['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'])

# Conver product_id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))